# 6. 데이터 전처리와 탐색

> **제6장** · **이론편 대응: 8.5절(일반화)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: **pandas** (대부분 이미 설치됨)
> **다운로드**: 없음 (데이터를 코드로 생성)

---

## 이 장에서 하는 일

다음 장부터 모델을 학습시킨다. 그런데 **모델보다 데이터가 먼저다.**

실무에서 시간의 대부분은 모델을 고르는 데가 아니라 **데이터를 다듬는 데** 쓰인다.
이 장은 그 과정을 다룬다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 준비 — pandas 확인 | — |
| 2 | 데이터 살펴보기 (EDA) | — |
| 3 | **결측치 — 처리 방법에 따라 성능이 달라진다** ★ | — |
| 4 | **이상치 — 찾는 것과 처리하는 것** | — |
| 5 | 스케일 문제 | 4장 4절 |
| 6 | 범주형 데이터 | — |
| 7 | **데이터 누수 — 가장 위험한 실수** ★ | 8.5절 |
| 8 | 전처리 파이프라인 | — |

**3절과 7절이 핵심이다.** 특히 7절의 데이터 누수는
**모델 성능이 실제보다 좋아 보이게 만드는** 함정이라 반드시 알아야 한다.

---

## 1. 준비 — pandas 확인

### pandas가 필요한 이유

2장에서 NumPy를 배웠다. NumPy는 **숫자 배열**을 다루는 데 최적이지만,
실제 데이터는 이렇지 않다.

| 상황 | NumPy | pandas |
|---|---|---|
| 열마다 이름이 있음 | 인덱스로 기억 | `df["나이"]` |
| 숫자와 문자가 섞임 | 어려움 | 자연스럽게 처리 |
| 결측치 | `np.nan` 수동 처리 | 전용 기능 |
| 그룹별 집계 | 직접 구현 | `groupby` |

**pandas는 NumPy 위에 만들어졌다.** 내부적으로는 NumPy 배열을 쓰므로,
2장에서 배운 것이 그대로 통한다.

In [ ]:
import importlib

print("=" * 60)
print("필요 패키지 확인")
print("=" * 60)

required = [
    ("numpy", "수치 계산 (2장)", True),
    ("pandas", "표 데이터", True),
    ("matplotlib", "시각화 (3장)", True),
    ("sklearn", "전처리 도구", True),
]

missing = []
for name, desc, essential in required:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "설치됨")
        print(f"[OK]   {name:<16}{ver:<14}{desc}")
    except ImportError:
        print(f"[없음] {name:<16}{'':<14}{desc}")
        missing.append(name)

print("-" * 60)
if missing:
    print(f"설치 필요: pip install {' '.join(missing)}")
else:
    print("[준비 완료] 2절로 진행하세요.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 100)
np.set_printoptions(precision=4, suppress=True)

print(f"pandas {pd.__version__} / numpy {np.__version__}")

---

## 2. 데이터 살펴보기

**실습용 데이터를 만든다.** 실제 데이터에서 흔히 마주치는 문제들을
일부러 심어 두었다 — 결측치, 이상치, 스케일 차이가 모두 들어 있다.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 200

# 직원 승진 예측 데이터를 흉내 낸다
age = np.random.randint(20, 65, N).astype(float)
income = np.random.lognormal(10.5, 0.6, N)          # 소득은 대체로 로그정규분포
years = np.clip(age - 22 + np.random.randn(N) * 3, 0, None)

department = np.random.choice(["개발", "영업", "관리", "연구"],
                              N, p=[0.4, 0.3, 0.2, 0.1])
education = np.random.choice(["학사", "석사", "박사"], N, p=[0.6, 0.3, 0.1])

# 목표 변수 — 여러 요인의 조합
score = (0.3 * (age - 40) / 12 +
         0.5 * np.log(income) / 10 +
         0.2 * years / 15 +
         np.random.randn(N) * 0.3)
promoted = (score > np.median(score)).astype(int)

df = pd.DataFrame({
    "나이": age,
    "소득": income,
    "근속연수": years,
    "부서": department,
    "학력": education,
    "승진": promoted,
})

# ── 실제 데이터처럼 만들기 ──
# 1) 결측치 주입
na_idx = np.random.choice(N, 25, replace=False)
df.loc[na_idx[:15], "소득"] = np.nan
df.loc[na_idx[15:], "근속연수"] = np.nan

# 2) 이상치 주입 (입력 오류를 흉내)
out_idx = np.random.choice(N, 5, replace=False)
df.loc[out_idx, "소득"] *= 12

print("=" * 70)
print("데이터 생성 완료")
print("=" * 70)
print(f"모양: {df.shape}   ← ({df.shape[0]}행, {df.shape[1]}열)")
print()
print(df.head(8))

In [ ]:
print("=" * 78)
print("첫 단계: 무엇이 들어 있나")
print("=" * 78)
print()
print("[1] 자료형과 결측")
print(df.info())
print()

print("[2] 숫자 열의 요약 통계")
print(df.describe().round(1))
print()

print("[3] 범주형 열")
for col in ["부서", "학력"]:
    print(f"\n  {col}")
    counts = df[col].value_counts()
    for k, v in counts.items():
        bar = "█" * int(v / 3)
        print(f"    {k:<8}{v:>4}개  {v/len(df)*100:>5.1f}%  {bar}")

print()
print("[4] 목표 변수 균형")
print(f"  승진 = 1: {(df['승진']==1).sum()}개")
print(f"  승진 = 0: {(df['승진']==0).sum()}개")
print()
print("  균형이 맞다. 한쪽이 90% 이상이면 불균형 문제를 고려해야 한다 (7장).")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

for ax, col in zip(axes, ["나이", "소득", "근속연수"]):
    data = df[col].dropna()
    ax.hist(data, bins=25, color="#1E40AF", edgecolor="white")
    ax.axvline(data.mean(), color="#DC2626", linewidth=2, label=f"평균 {data.mean():.0f}")
    ax.axvline(data.median(), color="#0D9488", linewidth=2,
               linestyle="--", label=f"중앙값 {data.median():.0f}")
    ax.set_xlabel(col)
    ax.set_ylabel("빈도")
    ax.set_title(f"{col} 분포")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 70)
print("분포에서 읽을 것")
print("=" * 70)
for col in ["나이", "소득", "근속연수"]:
    d = df[col].dropna()
    skew = ((d - d.mean()) ** 3).mean() / d.std() ** 3
    gap = (d.mean() - d.median()) / d.std()
    print(f"\n  {col}")
    print(f"    평균 {d.mean():>10.1f}  중앙값 {d.median():>10.1f}")
    print(f"    비대칭도(왜도) {skew:>6.2f}", end="")
    if abs(skew) < 0.5:
        print("  → 대체로 대칭")
    elif skew > 0:
        print("  → 오른쪽으로 긴 꼬리 (큰 값이 드물게 있음)")
    else:
        print("  → 왼쪽으로 긴 꼬리")

print()
print("소득의 평균이 중앙값보다 훨씬 크다.")
print("  일부 큰 값이 평균을 끌어올린 것이다 — 4절에서 다룬다.")

---

## 3. 결측치 ★

**빠진 값을 어떻게 할 것인가.** 답이 하나가 아니고, **선택에 따라 결과가 달라진다.**

In [ ]:
import numpy as np
import pandas as pd

print("=" * 70)
print("결측치 현황")
print("=" * 70)

na_count = df.isna().sum()
na_pct = df.isna().mean() * 100

print(f"{'열':<12}{'결측 개수':<12}{'비율':<12}{'막대'}")
print("-" * 70)
for col in df.columns:
    bar = "█" * int(na_pct[col])
    print(f"{col:<12}{na_count[col]:<12}{na_pct[col]:>6.1f}%     {bar}")
print("-" * 70)
print()
print(f"결측이 하나라도 있는 행: {df.isna().any(axis=1).sum()}개 "
      f"({df.isna().any(axis=1).mean()*100:.1f}%)")
print()
print("[판단 기준]")
print("  5% 미만  : 삭제해도 큰 손실 없음")
print("  5~30%    : 대치를 고려")
print("  30% 이상 : 그 열을 쓸지 자체를 재검토")

In [ ]:
import numpy as np
import pandas as pd

print("=" * 78)
print("결측 처리 방법들")
print("=" * 78)

methods = {}

# 1) 행 삭제
methods["행 삭제"] = df.dropna().copy()

# 2) 평균 대치
d = df.copy()
for col in ["소득", "근속연수"]:
    d[col] = d[col].fillna(d[col].mean())
methods["평균 대치"] = d

# 3) 중앙값 대치
d = df.copy()
for col in ["소득", "근속연수"]:
    d[col] = d[col].fillna(d[col].median())
methods["중앙값 대치"] = d

# 4) 그룹별 중앙값 (부서마다 다르게)
d = df.copy()
for col in ["소득", "근속연수"]:
    d[col] = d.groupby("부서")[col].transform(lambda s: s.fillna(s.median()))
methods["부서별 중앙값"] = d

print(f"{'방법':<18}{'표본 수':<12}{'소득 평균':<16}{'소득 표준편차'}")
print("-" * 78)
for name, data in methods.items():
    print(f"{name:<18}{len(data):<12}{data['소득'].mean():<16.0f}"
          f"{data['소득'].std():.0f}")
print("-" * 78)
print()
print("주목할 점")
print("  '행 삭제'는 표본이 줄어든다 — 데이터가 적으면 큰 손실")
print("  '평균 대치'는 분산을 줄인다 — 모두 같은 값으로 채우므로")
print("  '그룹별'은 맥락을 반영하지만 그룹이 작으면 불안정")

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

print("=" * 78)
print("어느 방법이 나은가 — 실제로 학습시켜 비교")
print("=" * 78)
print()
print("같은 조건에서 결측 처리만 바꿔 성능을 잰다.")
print()

results = {}
for name, data in methods.items():
    X = data[["나이", "소득", "근속연수"]].values
    y = data["승진"].values

    # ── train_test_split 파라미터 ────────────────────────────────

    #   test_size     시험 데이터 비율.  기본값 0.25

    #                 예: 0.2(20%) / 0.3(30%) — 데이터가 적으면 0.2~0.3

    #   train_size    학습 비율. test_size 를 주면 생략 가능

    #   random_state  난수 시드.  기본값 None(매번 달라짐)

    #                 재현성을 위해 반드시 고정한다. 예: 42, 0, 2024

    #   shuffle       섞을지 여부.  기본값 True

    #                 시계열 데이터는 False (미래 정보 누수 방지)

    #   stratify      계층 추출 기준.  기본값 None

    #                 분류 문제는 stratify=y 로 클래스 비율을 유지한다

    # ──────────────────────────────────────────────────────────────

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y)

    # ── StandardScaler 파라미터 ──────────────────────────────────

    #   with_mean     평균을 0으로 맞출지.  기본값 True

    #                 희소 행렬에는 False (0이 아닌 값으로 채워지는 것 방지)

    #   with_std      표준편차를 1로 맞출지.  기본값 True

    #

    #   [중요] fit 은 학습 데이터로만, 시험 데이터는 transform 만 한다

    #          scaler.fit_transform(X_train)   ← 통계를 여기서 계산

    #          scaler.transform(X_test)        ← 그 통계를 그대로 적용

    #          전체로 fit 하면 데이터 누수가 된다 (7절)

    # ──────────────────────────────────────────────────────────────

    scaler = StandardScaler().fit(X_tr)
    # ── LogisticRegression 파라미터 ──────────────────────────────
    #   C             정규화 강도의 **역수**.  기본값 1.0
    #                 작을수록 강한 정규화. 예: 0.01(강함) / 100(약함)
    #   penalty       정규화 종류.  기본값 'l2'
    #                 'l1'(희소) / 'l2'(기본) / 'elasticnet' / None
    #   solver        최적화 알고리즘.  기본값 'lbfgs'
    #                 'liblinear'(작은 데이터) / 'saga'(큰 데이터, l1 지원)
    #   max_iter      최대 반복 횟수.  기본값 100
    #                 수렴 경고가 뜨면 1000~5000 으로 올린다
    #   class_weight  클래스 가중치.  기본값 None
    #                 불균형 데이터는 'balanced' 를 쓴다
    # ──────────────────────────────────────────────────────────────
    model = LogisticRegression(max_iter=1000)
    model.fit(scaler.transform(X_tr), y_tr)

    acc = accuracy_score(y_te, model.predict(scaler.transform(X_te)))
    results[name] = {"acc": acc, "n": len(data)}

print(f"{'방법':<18}{'표본 수':<12}{'시험 정확도':<14}{'막대'}")
print("-" * 78)
for name, r in results.items():
    bar = "█" * int(r["acc"] * 40)
    print(f"{name:<18}{r['n']:<12}{r['acc']:<14.4f}{bar}")
print("-" * 78)

best = max(results, key=lambda k: results[k]["acc"])
worst = min(results, key=lambda k: results[k]["acc"])
gap = results[best]["acc"] - results[worst]["acc"]
print()
print(f"최고: {best} ({results[best]['acc']:.4f})")
print(f"최저: {worst} ({results[worst]['acc']:.4f})")
print(f"차이: {gap:.4f}")
print()
print("[중요] 모델을 바꾸지 않았는데도 성능이 달라진다.")
print("  전처리는 '준비 작업'이 아니라 성능을 좌우하는 선택이다.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: 방법별 정확도 ---
ax = axes[0]
names = list(results.keys())
accs = [results[n]["acc"] for n in names]
ns = [results[n]["n"] for n in names]

colors = ["#0D9488" if a == max(accs) else "#94A3B8" for a in accs]
bars = ax.bar(range(len(names)), accs, color=colors)
for b, a in zip(bars, accs):
    ax.text(b.get_x()+b.get_width()/2, a+0.008, f"{a:.3f}", ha="center", fontsize=9)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=8, rotation=12, ha="right")
ax.set_ylabel("시험 정확도")
ax.set_ylim(min(accs)-0.05, max(accs)+0.05)
ax.set_title("결측 처리 방법에 따른 성능")
ax.grid(axis="y", alpha=0.3)

# --- 오른쪽: 대치 전후 분포 ---
ax = axes[1]
original = df["소득"].dropna()
mean_filled = methods["평균 대치"]["소득"]

ax.hist(original, bins=25, alpha=0.6, label="원본 (결측 제외)",
        color="#1E40AF", density=True)
ax.hist(mean_filled, bins=25, alpha=0.6, label="평균으로 채운 후",
        color="#EA580C", density=True)
ax.axvline(original.mean(), color="#DC2626", linewidth=2,
           linestyle="--", label="평균값")
ax.set_xlabel("소득")
ax.set_ylabel("밀도")
ax.set_xlim(0, original.quantile(0.97))
ax.set_title("평균 대치가 분포를 바꾼다")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("오른쪽 그래프에서 평균 위치에 막대가 솟은 것이 보인다.")
print("  15개를 모두 같은 값으로 채웠기 때문이다.")
print()
print(f"  원본 표준편차    : {original.std():.0f}")
print(f"  평균 대치 후     : {mean_filled.std():.0f}")
print("  → 분산이 줄어든다. 데이터가 실제보다 '균일해 보이게' 된다.")

### 결측이 무작위가 아닐 때

**결측 자체가 정보일 수 있다.**

예를 들어 "소득" 항목이 비어 있는 사람들이 특정 부서에 몰려 있다면,
그 사실 자체가 예측에 쓸 수 있는 신호다.

이런 경우 **"결측이었는지"를 별도 열로 남기는** 방법이 있다.

In [ ]:
import pandas as pd
import numpy as np

print("=" * 70)
print("결측이 무작위인지 확인")
print("=" * 70)

# 부서별 결측 비율
na_by_dept = df.groupby("부서").apply(
    lambda g: g["소득"].isna().mean() * 100, include_groups=False)

print(f"{'부서':<10}{'소득 결측 비율'}")
print("-" * 70)
for dept, pct in na_by_dept.items():
    bar = "█" * int(pct * 2)
    print(f"{dept:<10}{pct:>6.1f}%   {bar}")
print("-" * 70)
print()

spread = na_by_dept.max() - na_by_dept.min()
if spread > 10:
    print(f"부서 간 차이가 {spread:.1f}%p 로 크다 — 무작위가 아닐 가능성")
else:
    print(f"부서 간 차이가 {spread:.1f}%p 로 작다 — 무작위에 가까움")
print()

# 결측 표시 열 추가
df_flag = df.copy()
df_flag["소득_결측"] = df["소득"].isna().astype(int)
df_flag["소득"] = df_flag["소득"].fillna(df_flag["소득"].median())
df_flag["근속연수"] = df_flag["근속연수"].fillna(df_flag["근속연수"].median())

print("결측 표시 열을 추가한 방식")
print(df_flag[["소득", "소득_결측", "승진"]].head(6))
print()
print("이렇게 하면 모델이 '값이 없었다'는 사실도 학습에 쓸 수 있다.")

---

## 4. 이상치

2절에서 소득의 평균이 중앙값보다 훨씬 컸다. **아주 큰 값 몇 개** 때문이다.

이상치는 두 종류로 나눠 생각해야 한다.

| 종류 | 예 | 처리 |
|---|---|---|
| **오류** | 나이 250세, 소득 음수 | 제거하거나 수정 |
| **실제 극단값** | 임원의 높은 소득 | **함부로 지우면 안 됨** |

**구별이 어렵다는 것이 문제다.**

In [ ]:
import numpy as np
import pandas as pd

print("=" * 78)
print("이상치 탐지 — 두 가지 방법")
print("=" * 78)


def detect_iqr(series, k=1.5):
    """사분위 범위 방법 (이론편 6.2절)

    Q1 - k*IQR 보다 작거나 Q3 + k*IQR 보다 크면 이상치로 본다.
    k=1.5 가 관행이며, 3.0 이면 더 극단적인 것만 잡는다.
    """
    s = series.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    mask = (series < lower) | (series > upper)
    return mask.fillna(False), lower, upper


def detect_zscore(series, threshold=3.0):
    """표준점수 방법

    평균에서 표준편차의 몇 배 떨어져 있는지로 판단한다.
    정규분포를 가정하므로 치우친 분포에서는 부정확하다.
    """
    s = series.dropna()
    z = (series - s.mean()) / s.std()
    return (z.abs() > threshold).fillna(False), z


print(f"{'열':<12}{'IQR 방법':<16}{'Z-score 방법':<18}{'IQR 경계'}")
print("-" * 78)
for col in ["나이", "소득", "근속연수"]:
    mask_iqr, lo, hi = detect_iqr(df[col])
    mask_z, _ = detect_zscore(df[col])
    print(f"{col:<12}{mask_iqr.sum():<16}{mask_z.sum():<18}"
          f"[{lo:,.0f}, {hi:,.0f}]")
print("-" * 78)
print()
print("두 방법의 결과가 다르다.")
print("  IQR    : 분포 모양에 영향을 덜 받음 (중앙값 기준)")
print("  Z-score: 정규분포 가정 — 치우친 분포에서는 이상치를 놓치기 쉽다")
print()
print("소득처럼 오른쪽 꼬리가 긴 분포에는 IQR 이 낫다.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ["나이", "소득", "근속연수"]):
    data = df[col].dropna()
    bp = ax.boxplot(data, vert=True, patch_artist=True, widths=0.5)
    bp["boxes"][0].set_facecolor("#1E40AF")
    bp["boxes"][0].set_alpha(0.6)
    for flier in bp["fliers"]:
        flier.set(marker="o", color="#DC2626", markersize=5, alpha=0.7)

    mask, lo, hi = detect_iqr(df[col])
    ax.set_title(f"{col}\n이상치 {mask.sum()}개", fontsize=10)
    ax.set_xticks([])
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 70)
print("상자 그림(box plot) 읽는 법")
print("=" * 70)
print("  상자      : 25%~75% 구간 (IQR)")
print("  가운데 선  : 중앙값")
print("  수염      : 1.5 x IQR 범위")
print("  점        : 이상치로 판정된 값")
print()
print("소득에만 점이 많다. 오른쪽 꼬리가 긴 분포이기 때문이다.")

In [ ]:
import numpy as np
import pandas as pd

print("=" * 78)
print("이상치 처리 방법 비교")
print("=" * 78)

col = "소득"
mask, lo, hi = detect_iqr(df[col])
original = df[col].dropna()

# 1) 제거
removed = original[~detect_iqr(original)[0]]

# 2) 절단 (clipping) — 경계값으로 대체
clipped = original.clip(lower=lo, upper=hi)

# 3) 로그 변환 — 분포 자체를 바꾼다
logged = np.log1p(original)

print(f"{'방법':<16}{'표본 수':<12}{'평균':<14}{'표준편차':<14}{'왜도'}")
print("-" * 78)


def skew(s):
    return float(((s - s.mean()) ** 3).mean() / s.std() ** 3)


for name, s in [("원본", original), ("이상치 제거", removed),
                ("절단(clip)", clipped), ("로그 변환", logged)]:
    print(f"{name:<16}{len(s):<12}{s.mean():<14.1f}{s.std():<14.1f}{skew(s):.2f}")
print("-" * 78)
print()
print("로그 변환의 왜도가 0에 가장 가깝다 = 대칭에 가까워졌다.")
print()
print("[각 방법의 특징]")
print("  제거   : 정보가 사라진다. 정말 오류일 때만.")
print("  절단   : 표본은 유지하되 극단값을 완화. 순위 정보는 보존.")
print("  로그   : 분포를 바꿔 극단값의 영향을 줄인다. 해석이 달라짐에 주의.")
print()
print("소득·매출·조회수처럼 곱셈적으로 늘어나는 값에는 로그 변환이 잘 맞는다.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

ax = axes[0]
ax.hist(original, bins=30, color="#DC2626", edgecolor="white")
ax.set_title(f"원본 (왜도 {skew(original):.2f})")
ax.set_xlabel("소득")
ax.grid(alpha=0.3)

ax = axes[1]
ax.hist(clipped, bins=30, color="#EA580C", edgecolor="white")
ax.set_title(f"절단 (왜도 {skew(clipped):.2f})")
ax.set_xlabel("소득")
ax.grid(alpha=0.3)

ax = axes[2]
ax.hist(logged, bins=30, color="#0D9488", edgecolor="white")
ax.set_title(f"로그 변환 (왜도 {skew(logged):.2f})")
ax.set_xlabel("log(소득)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("오른쪽 그림이 종 모양에 가깝다.")
print("  많은 통계 방법과 모델이 이런 분포에서 더 잘 작동한다.")

---

## 5. 스케일 문제 — 4장 4절과 이어짐

4장에서 **표준화를 하지 않아 학습이 발산**하는 것을 봤다.
이 데이터에도 같은 문제가 있다.

In [ ]:
import numpy as np

print("=" * 78)
print("열마다 범위가 다르다")
print("=" * 78)
print(f"{'열':<12}{'최소':<14}{'최대':<16}{'범위':<16}{'표준편차'}")
print("-" * 78)
for col in ["나이", "소득", "근속연수"]:
    s = df[col].dropna()
    print(f"{col:<12}{s.min():<14.0f}{s.max():<16,.0f}"
          f"{s.max()-s.min():<16,.0f}{s.std():,.0f}")
print("-" * 78)
print()

ratio = df["소득"].dropna().std() / df["나이"].std()
print(f"소득의 표준편차가 나이의 {ratio:,.0f}배")
print()
print("이대로 학습시키면 어떻게 되나")
print("  거리 기반 방법(K-Means, KNN): 소득이 거의 모든 것을 결정")
print("  경사하강법: 스케일이 큰 방향으로 그래디언트가 지배 (4장 5절)")
print("  정규화 항이 있는 모델: 큰 값을 가진 특성이 불리하게 취급됨")

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

print("=" * 78)
print("스케일링 방법 비교")
print("=" * 78)

data = df[["나이", "소득", "근속연수"]].fillna(df.median(numeric_only=True))

scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
}

print(f"{'방법':<18}{'수식':<34}{'특징'}")
print("-" * 78)
print(f"{'StandardScaler':<18}{'(x - 평균) / 표준편차':<34}{'평균 0, 분산 1'}")
print(f"{'MinMaxScaler':<18}{'(x - 최소) / (최대 - 최소)':<34}{'0~1 범위'}")
print(f"{'RobustScaler':<18}{'(x - 중앙값) / IQR':<34}{'이상치에 강함'}")
print("-" * 78)
print()

for name, scaler in scalers.items():
    scaled = scaler.fit_transform(data)
    print(f"\n[{name}]")
    print(f"{'열':<12}{'평균':<12}{'표준편차':<12}{'최소':<12}{'최대'}")
    for i, col in enumerate(data.columns):
        v = scaled[:, i]
        print(f"{col:<12}{v.mean():<12.3f}{v.std():<12.3f}"
              f"{v.min():<12.2f}{v.max():.2f}")

print()
print("-" * 78)
print("소득처럼 이상치가 있는 열을 보라.")
print("  MinMaxScaler: 이상치 때문에 대부분의 값이 0 근처에 몰린다")
print("  RobustScaler: 중앙값과 IQR 을 쓰므로 영향이 적다")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))

col_idx = 1      # 소득
titles = ["원본"] + list(scalers.keys())
datasets = [data.values[:, col_idx]]
for scaler in scalers.values():
    datasets.append(scaler.fit_transform(data)[:, col_idx])

colors = ["#DC2626", "#1E40AF", "#EA580C", "#0D9488"]

for ax, title, d, color in zip(axes, titles, datasets, colors):
    ax.hist(d, bins=30, color=color, edgecolor="white")
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_xlabel("소득 (변환 후)" if title != "원본" else "소득")

plt.tight_layout()
plt.show()

print("MinMaxScaler 결과를 보라 — 대부분이 왼쪽 끝에 몰려 있다.")
print("  최댓값이 이상치라서 나머지가 압축된 것이다.")
print()
print("[선택 기준]")
print("  일반적인 경우        → StandardScaler")
print("  범위가 정해져야 할 때 → MinMaxScaler (신경망 입력 등)")
print("  이상치가 있을 때     → RobustScaler")

---

## 6. 범주형 데이터

"부서"와 "학력"은 숫자가 아니다. **모델은 숫자만 다루므로** 변환이 필요하다.

In [ ]:
import pandas as pd
import numpy as np

print("=" * 78)
print("범주형 → 숫자 변환")
print("=" * 78)

sample = df[["부서", "학력"]].head(6)
print("\n원본")
print(sample)

# ── 1) 순서 인코딩 (Ordinal) ──
print("\n\n[방법 1] 순서 인코딩 — 순서가 있을 때")
edu_order = {"학사": 0, "석사": 1, "박사": 2}
sample_ord = sample.copy()
sample_ord["학력_숫자"] = sample["학력"].map(edu_order)
print(sample_ord[["학력", "학력_숫자"]])
print()
print("  학력은 순서가 있으므로 이 방법이 자연스럽다.")

# ── 2) 원-핫 인코딩 ──
print("\n\n[방법 2] 원-핫 인코딩 — 순서가 없을 때")
dummies = pd.get_dummies(df["부서"], prefix="부서").astype(int)
print(dummies.head(6))
print()
print("  부서에는 순서가 없다.")
print("  '개발=0, 영업=1, 관리=2' 로 하면 모델이 '관리 > 영업 > 개발' 로 오해한다.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

print("=" * 78)
print("잘못된 인코딩의 영향")
print("=" * 78)

base = df.copy()
for c in ["소득", "근속연수"]:
    base[c] = base[c].fillna(base[c].median())

# 잘못된 방법: 순서 없는 범주에 숫자 부여
wrong = base.copy()
wrong["부서"] = wrong["부서"].map({"개발": 0, "영업": 1, "관리": 2, "연구": 3})
X_wrong = wrong[["나이", "소득", "근속연수", "부서"]].values

# 올바른 방법: 원-핫
right = pd.concat([
    base[["나이", "소득", "근속연수"]],
    pd.get_dummies(base["부서"], prefix="부서").astype(int),
], axis=1)
X_right = right.values

y = base["승진"].values

print(f"{'방법':<20}{'특성 수':<12}{'시험 정확도'}")
print("-" * 78)
for name, X in [("순서 부여 (잘못)", X_wrong), ("원-핫 (올바름)", X_right)]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y)
    sc = StandardScaler().fit(X_tr)
    m = LogisticRegression(max_iter=1000).fit(sc.transform(X_tr), y_tr)
    acc = accuracy_score(y_te, m.predict(sc.transform(X_te)))
    print(f"{name:<20}{X.shape[1]:<12}{acc:.4f}")
print("-" * 78)
print()
print("이 데이터에서는 차이가 크지 않을 수 있다.")
print("  부서가 목표와 관련이 약하게 설계되었기 때문이다.")
print()
print("하지만 범주가 많고 관련성이 클수록 차이가 커진다.")
print("  원칙: **순서가 없으면 원-핫**")
print()
print("[주의] 범주가 아주 많을 때")
print("  원-핫은 열이 범주 수만큼 늘어난다.")
print("  범주가 100개면 열이 100개 추가 — 차원의 저주 (이론편 4.4절)")
print("  이럴 때는 빈도 인코딩, 타깃 인코딩, 임베딩(27번) 등을 쓴다.")

---

## 7. 데이터 누수 ★ — 이론편 8.5절

**가장 위험한 실수**를 다룬다. 모델 성능이 실제보다 좋아 보이게 만드는 함정이다.

$$\text{데이터 누수} = \text{학습 시점에 알 수 없는 정보가 섞여 들어가는 것}$$

가장 흔한 형태는 **전처리를 나누기 전에 하는 것**이다.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("=" * 78)
print("전형적인 실수: 나누기 전에 스케일링")
print("=" * 78)
print()

X = base[["나이", "소득", "근속연수"]].values
y = base["승진"].values

# ── 잘못된 방법 ──
print("[잘못된 방법]")
print("  1) 전체 데이터로 StandardScaler 를 fit")
print("  2) 그다음 학습/시험으로 분리")
print()

scaler_wrong = StandardScaler()
X_scaled_all = scaler_wrong.fit_transform(X)          # 전체로 fit — 여기가 문제
X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(
    X_scaled_all, y, test_size=0.3, random_state=42, stratify=y)

m_wrong = LogisticRegression(max_iter=1000).fit(X_tr_w, y_tr_w)
acc_wrong = accuracy_score(y_te_w, m_wrong.predict(X_te_w))

# ── 올바른 방법 ──
print("[올바른 방법]")
print("  1) 먼저 학습/시험으로 분리")
print("  2) 학습 데이터로만 fit, 시험 데이터에는 transform 만")
print()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

scaler_right = StandardScaler().fit(X_tr)             # 학습 데이터로만 fit
m_right = LogisticRegression(max_iter=1000).fit(scaler_right.transform(X_tr), y_tr)
acc_right = accuracy_score(y_te, m_right.predict(scaler_right.transform(X_te)))

print("-" * 78)
print(f"{'방법':<28}{'시험 정확도':<16}{'비고'}")
print("-" * 78)
print(f"{'전체로 스케일링 (누수)':<28}{acc_wrong:<16.4f}실제보다 좋아 보일 수 있음")
print(f"{'학습으로만 스케일링':<28}{acc_right:<16.4f}올바른 추정")
print("-" * 78)
print()
print("무엇이 새어 나갔나")
print("  StandardScaler 는 평균과 표준편차를 계산한다.")
print("  전체로 fit 하면 **시험 데이터의 통계가 학습에 반영**된다.")
print()
print(f"  전체 평균 : {X[:, 1].mean():>12,.0f}")
print(f"  학습 평균 : {X_tr[:, 1].mean():>12,.0f}")
print(f"  차이      : {abs(X[:, 1].mean() - X_tr[:, 1].mean()):>12,.0f}")

In [ ]:
print("=" * 78)
print("데이터 누수의 여러 형태 (이론편 8.5절)")
print("=" * 78)
print()
print(f"{'형태':<26}{'예':<30}{'해결'}")
print("-" * 78)
leaks = [
    ("전처리 순서",      "전체로 스케일링 후 분리",       "분리 후 학습으로만 fit"),
    ("결측 대치",        "전체 평균으로 채운 뒤 분리",     "학습 평균으로 채우기"),
    ("미래 정보",        "시계열에서 나중 데이터 사용",     "시간 순서로 분리"),
    ("중복 행",          "같은 행이 학습·시험 양쪽에",     "분리 전 중복 제거"),
    ("목표 파생 특성",   "'승진 여부'로 만든 특성 사용",    "특성 생성 시점 확인"),
    ("그룹 누수",        "같은 사람의 여러 기록이 분산",    "GroupKFold 사용"),
]
for a, b, c in leaks:
    print(f"{a:<26}{b:<30}{c}")
print("-" * 78)
print()
print("[증상]")
print("  검증 성능은 아주 좋은데 실제 운영에서 뚝 떨어진다.")
print("  '너무 좋은 결과'가 나오면 누수를 먼저 의심해야 한다.")
print()
print("[예방]")
print("  전처리를 Pipeline 으로 묶으면 실수를 원천적으로 막을 수 있다 (8절).")

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("=" * 78)
print("교차검증에서의 누수 — 더 미묘한 경우")
print("=" * 78)
print()

X = base[["나이", "소득", "근속연수"]].values
y = base["승진"].values
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 잘못: 미리 전체 스케일링 후 교차검증
X_pre_scaled = StandardScaler().fit_transform(X)
scores_leak = cross_val_score(
    LogisticRegression(max_iter=1000), X_pre_scaled, y, cv=cv)

# 올바름: Pipeline 안에 스케일러를 넣어 폴드마다 fit
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
scores_ok = cross_val_score(pipe, X, y, cv=cv)

print(f"{'방식':<32}{'평균 정확도':<16}{'표준편차'}")
print("-" * 78)
print(f"{'미리 스케일링 (누수)':<32}{scores_leak.mean():<16.4f}{scores_leak.std():.4f}")
print(f"{'Pipeline (올바름)':<32}{scores_ok.mean():<16.4f}{scores_ok.std():.4f}")
print("-" * 78)
print()
print("차이가 작아 보여도 이 데이터가 단순하기 때문이다.")
print("  특성이 많고 복잡한 데이터일수록 격차가 커진다.")
print()
print("[핵심] Pipeline 을 쓰면 각 폴드마다 스케일러가 새로 fit 된다.")
print("  즉 검증 폴드의 정보가 절대 학습에 섞이지 않는다.")

---

## 8. 전처리 파이프라인

지금까지의 처리를 **하나로 묶는다.** 7절에서 봤듯 Pipeline은 누수를 막아 주고,
같은 처리를 새 데이터에 그대로 적용할 수 있게 해 준다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report

print("=" * 78)
print("전처리 파이프라인 구성")
print("=" * 78)

# 열을 성격별로 나눈다
numeric_cols = ["나이", "소득", "근속연수"]
ordinal_cols = ["학력"]
nominal_cols = ["부서"]

# 각 성격에 맞는 처리
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),      # 3절
    ("scale", StandardScaler()),                        # 5절
])

ordinal_pipe = Pipeline([
    ("encode", OrdinalEncoder(categories=[["학사", "석사", "박사"]])),  # 6절
])

nominal_pipe = Pipeline([
    ("encode", OneHotEncoder(drop="first", sparse_output=False)),       # 6절
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("ord", ordinal_pipe, ordinal_cols),
    ("nom", nominal_pipe, nominal_cols),
])

full_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

print("\n구성")
print(f"  숫자형 {numeric_cols}")
print(f"    → 중앙값 대치 → 표준화")
print(f"  순서형 {ordinal_cols}")
print(f"    → 순서 인코딩 (학사 < 석사 < 박사)")
print(f"  명목형 {nominal_cols}")
print(f"    → 원-핫 인코딩 (첫 범주 제외)")
print()

X = df[numeric_cols + ordinal_cols + nominal_cols]
y = df["승진"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

full_pipeline.fit(X_tr, y_tr)
acc = accuracy_score(y_te, full_pipeline.predict(X_te))

print(f"시험 정확도: {acc:.4f}")
print()

# 변환 후 특성 확인
feature_names = full_pipeline.named_steps["prep"].get_feature_names_out()
print(f"변환 후 특성 {len(feature_names)}개")
for name in feature_names:
    print(f"  {name}")

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold

print("=" * 78)
print("파이프라인의 장점")
print("=" * 78)
print()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(full_pipeline, X, y, cv=cv, scoring="accuracy")

print("[1] 교차검증이 안전하다")
print(f"  5-폴드 정확도: {scores.round(4)}")
print(f"  평균 {scores.mean():.4f} (표준편차 {scores.std():.4f})")
print("  각 폴드마다 전처리가 새로 fit 되므로 누수가 없다.")
print()

print("[2] 새 데이터에 그대로 적용된다")
new_data = X.head(3).copy()
new_data.loc[new_data.index[0], "소득"] = np.nan       # 결측이 있어도
pred = full_pipeline.predict(new_data)
proba = full_pipeline.predict_proba(new_data)[:, 1]
print(f"  새 데이터 3건 예측: {pred}")
print(f"  승진 확률: {proba.round(3)}")
print("  결측 대치·인코딩·스케일링이 자동으로 적용된다.")
print()

print("[3] 저장하고 불러올 수 있다")
print("  import joblib")
print("  joblib.dump(full_pipeline, 'model.pkl')")
print("  loaded = joblib.load('model.pkl')")
print("  → 전처리 방식까지 함께 저장된다")
print()
print("-" * 78)
print("[가장 중요한 점]")
print("  전처리를 손으로 하면 학습 때와 예측 때 다르게 할 위험이 있다.")
print("  Pipeline 은 그 가능성을 원천적으로 없앤다.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("=" * 78)
print("전처리 유무에 따른 성능 종합")
print("=" * 78)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_num = df[numeric_cols]
y_all = df["승진"]

configs = {
    "대치만": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", LogisticRegression(max_iter=1000)),
    ]),
    "대치 + 표준화": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000)),
    ]),
    "전체 (범주형 포함)": full_pipeline,
}

results_final = {}
for name, pipe in configs.items():
    data_X = X if name == "전체 (범주형 포함)" else X_num
    s = cross_val_score(pipe, data_X, y_all, cv=cv, scoring="accuracy")
    results_final[name] = s

print(f"{'구성':<24}{'평균 정확도':<16}{'표준편차':<14}{'폴드별'}")
print("-" * 78)
for name, s in results_final.items():
    print(f"{name:<24}{s.mean():<16.4f}{s.std():<14.4f}{s.round(3)}")
print("-" * 78)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
names = list(results_final.keys())
means = [results_final[n].mean() for n in names]
stds = [results_final[n].std() for n in names]

bars = ax.bar(range(len(names)), means, yerr=stds, capsize=6,
              color=["#94A3B8", "#1E40AF", "#0D9488"])
for b, m in zip(bars, means):
    ax.text(b.get_x() + b.get_width()/2, m + 0.02, f"{m:.3f}",
            ha="center", fontsize=10)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel("교차검증 정확도")
ax.set_ylim(min(means) - 0.1, max(means) + 0.08)
ax.set_title("전처리 구성에 따른 성능 (오차 막대는 표준편차)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("모델은 그대로인데 전처리만 바꿔도 성능이 달라진다.")
print("  '어떤 모델을 쓸까'보다 '데이터를 어떻게 다듬을까'가 먼저다.")

---

## 9. 정리

### 처리 순서

```
1. 살펴보기      info(), describe(), 분포 그리기
2. 결측 처리     삭제 / 대치 / 결측 표시 추가
3. 이상치        IQR·Z-score 로 탐지 → 제거 / 절단 / 변환
4. 범주형 변환    순서 있으면 Ordinal, 없으면 One-Hot
5. 스케일링      Standard / MinMax / Robust
6. Pipeline 으로 묶기
```

**5장까지를 Pipeline 안에 넣는 것**이 핵심이다. 그래야 누수가 없다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| 결측 처리 | 방법에 따라 **성능이 달라진다** |
| 평균 대치 | 분산을 줄인다 — 데이터가 균일해 보이게 됨 |
| 결측 표시 | 결측 자체가 정보일 수 있다 |
| IQR vs Z-score | 치우친 분포에는 **IQR** |
| 이상치 | 오류인지 실제 극단값인지 구별해야 |
| 로그 변환 | 곱셈적으로 늘어나는 값(소득·매출)에 적합 |
| 범주형 | **순서 없으면 원-핫** |
| **데이터 누수** | **분리 후 학습 데이터로만 fit** |
| Pipeline | 누수 방지 + 재사용 + 저장 |

### 실무에서 자주 하는 실수

1. **전체 데이터로 스케일링한 뒤 분리** — 7절
2. 순서 없는 범주에 숫자 부여 — 6절
3. 이상치를 무조건 제거 — 4절
4. 결측을 무조건 평균으로 대치 — 3절
5. 전처리를 손으로 반복 — 8절

### 다음 장

**7. scikit-learn 지도학습** — 이제 다듬은 데이터로 모델을 학습시킨다.
이 장에서 만든 Pipeline이 그대로 쓰인다.